# Encoder–HistGB

The baseline is the unchanged pipeline from **features_selection.ipynb**:
current imputation, engineered features and tuned HistGB, without feature selection.
We only append one contextual embedding per visit.

The encoder receives the original imputed features, their missingness indicators and elapsed
time since the first visit. It uses two bidirectional Transformer encoder blocks, four heads,
width 64 and **no CLS token**. Each patient's available visits form one sequence.

Validation trains the whole pipeline separately in each patient split. There are **no
out-of-fold embeddings**. The outer validation patients never train the encoder or HistGB.
An internal patient holdout selects the epoch count; the encoder is then refitted on all
outer training patients for that many epochs.

## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Run this notebook from the project root or exploration folder.
PyTorch is required: uv pip install --python .venv/bin/python torch.

In [1]:
import json
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

NOTEBOOK_DIR = Path("exploration") if Path("exploration").is_dir() else Path(".")
reference = json.loads((NOTEBOOK_DIR / "features_selection.ipynb").read_text())
for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
    exec("".join(reference["cells"][i]["source"]))
baseline = clone(pipeline)

ENCODER = dict(width=64, heads=4, layers=2, dropout=0.1)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
SEEDS = [42, 43, 44]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. One sequence per patient

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [2]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Small bidirectional encoder

Dense input projection → two sequential TransformerEncoder layers → one embedding per visit.
Each layer already includes parallel attention heads, feed-forward layers, residual connections
and LayerNorm. The regression head is used for supervised training only.

In [3]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=2 * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        self.head = nn.Linear(width, 1)

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [4]:
def train_network(training, seed, epochs, validation=None):
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **ENCODER).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE,
                                 weight_decay=WEIGHT_DECAY)
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column.

In [5]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.head.in_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Fold-specific Encoder–HistGB validation

Use the same five patient-separated splits as the existing notebooks. Preprocessing is fitted
again in each split and shared by the unchanged baseline and augmented model. For each seed,
train a fresh encoder on that split's training patients, freeze it, append its embeddings and
fit a fresh clone of the exact same HistGB. No embeddings are combined across splits.
Negative delta_MSE means the embeddings help. Standard deviations describe repeated,
overlapping splits and encoder seeds; they are not independent confidence intervals.

In [ ]:
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        imputed_train = prep[:2].transform(raw_train)
        imputed_val = prep[:2].transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        baseline_mse = mean_squared_error(yv, original.predict(validation))

        for seed in SEEDS:
            fitted_encoder = fit_encoder(raw_train, yt, imputed_train, seed)
            z_train, _ = encoder_outputs(fitted_encoder, raw_train, imputed_train)
            z_val, encoder_prediction = encoder_outputs(fitted_encoder, raw_val, imputed_val)
            augmented = clone(baseline[-1]).fit(train.join(z_train), yt)
            mse = mean_squared_error(yv, augmented.predict(validation.join(z_val)))
            scores.append(dict(fold=fold, seed=seed, epochs=fitted_encoder["epochs"],
                               HistGB=baseline_mse, Transformer=mean_squared_error(yv, encoder_prediction),
                               Encoder_HistGB=mse, delta_MSE=mse - baseline_mse))
            print(f"Fold {fold}/5, seed {seed}, {fitted_encoder['epochs']} epochs: "
                  f"HistGB {baseline_mse:.4f} → Encoder–HistGB {mse:.4f}")
scores = pd.DataFrame(scores)
scores

Fold 1/5, seed 42, 63 epochs: HistGB 12.1961 → Encoder–HistGB 8.1232
Fold 1/5, seed 43, 58 epochs: HistGB 12.1961 → Encoder–HistGB 7.9687
Fold 1/5, seed 44, 52 epochs: HistGB 12.1961 → Encoder–HistGB 8.0052
Fold 2/5, seed 42, 32 epochs: HistGB 11.5942 → Encoder–HistGB 8.5310
Fold 2/5, seed 43, 35 epochs: HistGB 11.5942 → Encoder–HistGB 8.0245
Fold 2/5, seed 44, 42 epochs: HistGB 11.5942 → Encoder–HistGB 8.1770
Fold 3/5, seed 42, 40 epochs: HistGB 11.5700 → Encoder–HistGB 8.1780


In [ ]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

## 7. Optional final training and test predictions

After reviewing validation, set RUN_FINAL to True and choose a seed. A single encoder is
trained on all training patients and paired with a single HistGB using the same parameters.
The outer test data is transformed only; it never fits preprocessing or either model.

In [ ]:
RUN_FINAL = False
FINAL_SEED = 42

if RUN_FINAL:
    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    with threadpool_limits(limits=3):
        final_prep = clone(baseline[:-1])
        train = final_prep.fit_transform(X, y)
        test = final_prep.transform(X_test)
        imputed_train = final_prep[:2].transform(X)
        imputed_test = final_prep[:2].transform(X_test)
        final_encoder = fit_encoder(X, y, imputed_train, FINAL_SEED)
        z_train, _ = encoder_outputs(final_encoder, X, imputed_train)
        z_test, _ = encoder_outputs(final_encoder, X_test, imputed_test)
        final_histgb = clone(baseline[-1]).fit(train.join(z_train), y)
        prediction = final_histgb.predict(test.join(z_test))
    assert np.isfinite(prediction).all()
    submission = pd.DataFrame({"Index": X_test["Index"], "target": prediction})
    submission_path = DATA_DIR / "submission_encoder_histgb.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")

## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Vary encoder depth, width and heads, or replace the encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed HistGB the encoder's final prediction instead of, or alongside, its embeddings.